In [1]:
import os
os.chdir("..")   

import sys
sys.path.append("src")

import pandas as pd

from wildfire_gnn.pipelines.baseline_pipeline import BaselinePipeline
from wildfire_gnn.utils.config import load_yaml_config

In [3]:
import os
print(os.getcwd())

d:\wildfire-uncertainty-gnn


In [4]:
config = load_yaml_config("configs/baseline_config.yaml")
pipeline = BaselinePipeline(config)

In [5]:
results_df = pipeline.train_and_evaluate()
results_df

2026-04-01 14:51:05 | INFO | baseline_pipeline | Saved baseline dataset to data/processed/baseline_dataset.csv
2026-04-01 14:51:05 | INFO | baseline_pipeline | Baseline dataset stats: {'num_rows': 300000, 'num_columns': 10, 'target_min': 2.3405966203426942e-05, 'target_max': 0.24930262565612793, 'target_mean': 0.02515261061489582, 'target_std': 0.03347640857100487, 'missing_values_total': 0}
2026-04-01 14:51:05 | INFO | baseline_pipeline | Saved feature summary to reports\tables\baseline_feature_summary.csv
2026-04-01 14:51:06 | INFO | baseline_pipeline | Saved random splits to data/processed/baseline_splits_random.npz
2026-04-01 14:51:07 | INFO | baseline_pipeline | Saved spatial splits to data/processed/baseline_splits_spatial.npz
2026-04-01 14:51:07 | INFO | baseline_pipeline | [random] train_idx size=210000 target_mean=0.025191 target_std=0.033587
2026-04-01 14:51:07 | INFO | baseline_pipeline | [random] val_idx size=45000 target_mean=0.024981 target_std=0.033231
2026-04-01 14:51:0

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,rf,random,val,0.006088,0.003279,0.966430,0.983110,0.969646
1,rf,random,test,0.006041,0.003280,0.966905,0.983360,0.969912
2,mlp,random,val,0.016202,0.010322,0.762266,0.873848,0.822445
3,mlp,random,test,0.016427,0.010406,0.755243,0.870133,0.823740
4,xgb,random,val,0.009695,0.006241,0.914876,0.956919,0.918205
5,xgb,random,test,0.009761,0.006288,0.913576,0.956380,0.918772
6,rf,spatial,val,0.024677,0.015964,0.430115,0.709546,0.712069
7,rf,spatial,test,0.019917,0.012806,-0.654964,0.353729,0.447870
8,mlp,spatial,val,0.026350,0.017930,0.350247,0.631453,0.693358
9,mlp,spatial,test,0.021048,0.014448,-0.848325,0.441836,0.405759


In [6]:
metrics_df = pd.read_csv("reports/tables/baseline_metrics.csv")
metrics_df

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,rf,random,val,0.006088,0.003279,0.966430,0.983110,0.969646
1,rf,random,test,0.006041,0.003280,0.966905,0.983360,0.969912
2,mlp,random,val,0.016202,0.010322,0.762266,0.873848,0.822445
3,mlp,random,test,0.016427,0.010406,0.755243,0.870133,0.823740
4,xgb,random,val,0.009695,0.006241,0.914876,0.956919,0.918205
5,xgb,random,test,0.009761,0.006288,0.913576,0.956380,0.918772
6,rf,spatial,val,0.024677,0.015964,0.430115,0.709546,0.712069
7,rf,spatial,test,0.019917,0.012806,-0.654964,0.353729,0.447870
8,mlp,spatial,val,0.026350,0.017930,0.350247,0.631453,0.693358
9,mlp,spatial,test,0.021048,0.014448,-0.848325,0.441836,0.405759


In [7]:
feature_summary_df = pd.read_csv("reports/tables/baseline_feature_summary.csv")
feature_summary_df.head()

,feature,min,max,mean,std,n_unique
0,CFL.img,-11.182920,0.167812,3.967285e-09,1.000002,103
1,FSP_Index.img,-30.038822,0.072349,2.212524e-09,1.000002,1050
2,Ignition_Prob.img,-18.305900,0.096481,-9.663900e-09,1.000002,1613
3,Struct_Exp_Index.img,-30.038822,0.072349,2.212524e-09,1.000002,1050
4,Fuel_Models.img,91.000000,189.000000,1.373370e+02,29.096457,24


In [8]:
rf_preds = pd.read_csv("reports/tables/rf_test_predictions.csv")
rf_preds.head()

,row_index,col_index,y_true,y_pred,residual
0,1836,1247,0.017760,0.015816,0.001944
1,1798,1169,0.003567,0.003477,0.000090
2,1061,731,0.019104,0.019795,-0.000692
3,347,495,0.062656,0.058603,0.004054
4,615,220,0.000666,0.001453,-0.000786


#  Phase 4A — Tabular Baseline Models

##  Objective

The objective of Phase 4A is to establish **strong non-graph baseline models** for wildfire burn probability prediction. These baselines serve as a critical reference point for evaluating the effectiveness of graph-based models introduced in later phases.

This phase aims to answer the following key question:

> *How well can classical machine learning and feedforward neural networks model wildfire spread without explicitly incorporating spatial dependencies?*

---

##  Dataset Construction

The baseline dataset was derived from the constructed graph representation by treating each node as an independent sample.

- **Total samples:** 300,000 nodes  
- **Features:**
  - CFL (Canopy Fuel Load)
  - FSP Index
  - Ignition Probability
  - Structural Exposure Index
  - Fuel Models
- **Target:**
  - Burn probability (continuous)

---

###  Dataset Statistics

| Statistic | Value |
|----------|------|
| Number of samples | 300,000 |
| Number of features | 10 |
| Target min | 0.000023 |
| Target max | 0.2493 |
| Target mean | 0.02515 |
| Target std | 0.03347 |
| Missing values | 0 |

---

###  Key Observation: Target Distribution

The target distribution is **highly skewed toward low burn probabilities**, as shown in the histogram:

- Most values are near zero
- High burn probability events are rare

 This creates:
- **Imbalanced regression problem**
- Difficulty in learning extreme (high-risk) events

---

##  Data Splitting Strategy

To ensure robust evaluation, two different splitting strategies were used.

---

### 1️ Random Split (Standard ML Setting)

- Train: 70% (210,000 samples)
- Validation: 15% (45,000 samples)
- Test: 15% (45,000 samples)

 Assumes data is IID (independent and identically distributed)

---

### 2️ Spatial Block Split (Realistic Setting)

- Data is partitioned based on spatial coordinates
- Entire geographic regions assigned to train/val/test

👉 Prevents spatial leakage  
👉 Simulates real-world deployment where model sees unseen regions

---

###  Split Distribution (Important Insight)

| Split | Target Mean | Target Std |
|------|------------|-----------|
| Random Train | 0.02519 | 0.03359 |
| Random Test | 0.02515 | 0.03320 |
| Spatial Train | 0.02811 | 0.03642 |
| Spatial Test | 0.01157 | 0.01548 |

 **Significant distribution shift exists in spatial split**

---

##  Models Evaluated

###  Random Forest (RF)
- Ensemble of decision trees
- Handles nonlinear relationships well
- Robust to feature scaling

---

###  XGBoost (XGB)
- Gradient boosting model
- Strong tabular baseline
- Includes regularization

---

###  MLP (Feedforward Neural Network)
- Fully connected neural network
- Uses feature scaling
- No explicit spatial awareness

---

##  Evaluation Metrics

The following regression metrics were used:

- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)
- R² Score
- Pearson Correlation
- Spearman Correlation

---

#  Results

---

##  Random Split Performance

| Model | RMSE | MAE | R² | Pearson | Spearman |
|------|------|-----|-----|--------|----------|
| RF   | **0.00604** | **0.00328** | **0.9669** | **0.9833** | **0.9699** |
| XGB  | 0.00976 | 0.00629 | 0.9136 | 0.9564 | 0.9188 |
| MLP  | 0.01643 | 0.01041 | 0.7552 | 0.8701 | 0.8237 |

---

###  Interpretation

- **Random Forest achieves the best performance**
- Extremely high R² (~0.97)
- Very strong correlation (>0.98)

 At first glance, the problem appears easy

---

##  Critical Issue: Spatial Leakage

These results are **overly optimistic** due to spatial leakage:

- Nearby pixels share similar properties
- Random split allows spatial overlap between train and test

 Model indirectly learns geographic patterns

---

##  Spatial Split Performance (Realistic Scenario)

| Model | RMSE | MAE | R² | Pearson | Spearman |
|------|------|-----|-----|--------|----------|
| RF   | 0.01992 | 0.01281 | -0.65 | 0.35 | 0.45 |
| XGB  | **0.01733** | **0.01215** | -0.25 | 0.35 | 0.38 |
| MLP  | 0.02105 | 0.01445 | -0.84 | 0.44 | 0.41 |

---

##  Key Findings

### 1️ Performance Collapse

- RMSE increases ~3x
- R² becomes **negative**

 Models fail to generalize to unseen regions

---

### 2️ Negative R² (Important)

R² < 0 means:

> The model performs worse than predicting the mean

 Strong evidence of poor generalization

---

### 3️ XGBoost performs best under spatial split

- Lowest RMSE among baselines
- More robust to distribution shift

---

### 4️ MLP performs worst

- Highest error
- Unstable predictions
- Sensitive to data distribution

---

##  Visual Analysis

###  Target Distribution
- Highly skewed
- Long-tail behavior
- Imbalanced regression

---

###  Prediction vs True

- RF and XGB:
  - Strong alignment under random split
  - Degraded under spatial split
- MLP:
  - High variance
  - Poor fit for high values

---

###  Feature Importance

Top features:

- row_norm / col_norm (spatial proxies)
- Fuel Models

 Indicates models still rely on spatial patterns indirectly

---

#  Discussion

##  Why Tabular Models Fail

Tabular models assume:

- Independent samples

But wildfire spread is:

- Spatially dependent
- Influenced by neighbors

---

##  Missing Capability

Tabular models cannot:

- Capture spatial propagation
- Model neighbor interactions
- Represent graph structure

---

##  Core Insight

> High performance under random split is misleading. True difficulty emerges under spatial evaluation.

---

#  Implications for Next Phase

These results strongly justify moving to:

 **Graph Neural Networks (GNNs)**

Because GNNs:

- Explicitly model spatial relationships
- Enable message passing between nodes
- Capture wildfire spread dynamics

---

# Final Conclusions

###  Best Model (Random Split)
→ Random Forest

###  Best Model (Spatial Split)
→ XGBoost (still limited)

###  Weakest Model
→ MLP

---

##  Key Takeaway

> The wildfire prediction problem is fundamentally spatial, and tabular models fail to generalize under realistic conditions.

---

#  Paper-Ready Statement

> While classical machine learning models demonstrate strong predictive performance under random splits, their performance degrades significantly under spatially disjoint evaluation. This highlights the inability of tabular approaches to capture spatial dependencies inherent in wildfire dynamics, motivating the adoption of graph-based models in subsequent phases.